# 1. 模型训练

## 1.1 导入需要的库

In [2]:
import pandas as pd
import jieba
import re
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    cohen_kappa_score,
    mean_absolute_error,
    mean_squared_error
)

from scipy.stats import spearmanr
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC




In [3]:
df = pd.read_excel("D:/Desktop/Nanjing University/DSAI/dataset/人工评分.xlsx")
texts = df['弹幕内容'].astype(str).tolist()
labels = df['情绪评分'].astype(int).tolist()

## 1.2 进行分词

In [4]:
def chinese_tokenizer(text):
    return jieba.lcut(text)

vectorizer = TfidfVectorizer(tokenizer=chinese_tokenizer, max_features=5000)
X = vectorizer.fit_transform(texts)
y = labels

selector = SelectKBest(chi2, k=2000)
X_new = selector.fit_transform(X, y)


d:\Desktop\Nanjing University\DSAI\myenv\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\陈帅衡\AppData\Local\Temp\jieba.cache
Loading model cost 0.418 seconds.
Prefix dict has been built successfully.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X_new, y, test_size=0.2, random_state=42)

## 1.3 进行模型训练

In [6]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
y_pred_nb = nb_model.predict(X_test)

# 6.2 支持向量机
svm_model = SVC(kernel='linear', C=1, probability=True)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)


## 1.4 模型评估

In [26]:
def evaluate_model(y_true, y_pred, model_name):
    print(f"--- {model_name} ---")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1-score:", f1_score(y_true, y_pred))
    print("\n")

# 评估模型
evaluate_model(y_test, y_pred_nb, "Naive Bayes")
evaluate_model(y_test, y_pred_svm, "SVM")


--- Naive Bayes ---
Accuracy: 0.8946220930232558
Precision: 0.75
Recall: 0.02040816326530612
F1-score: 0.039735099337748346


--- SVM ---
Accuracy: 0.9018895348837209
Precision: 0.6578947368421053
Recall: 0.17006802721088435
F1-score: 0.2702702702702703




In [27]:
kappa = cohen_kappa_score(y_test, y_pred_nb)
print("Cohen's Kappa:", kappa)


Cohen's Kappa: 0.0342691190706681


In [8]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
from sklearn.metrics import r2_score

mae = mean_absolute_error(y_test, y_pred_nb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_nb))  # 手动开方

print("MAE_nb :", mae)
print("RMSE_nb:", rmse)
r2_score_nb = r2_score(y_test, y_pred_nb)
print("R2 Score_nb:", r2_score_nb)

mae_svm = mean_absolute_error(y_test, y_pred_svm)
rmse_svm = np.sqrt(mean_squared_error(y_test, y_pred_svm))
print("MAE_svm :", mae_svm)
print("RMSE_svm:", rmse_svm)
r2_score_svm = r2_score(y_test, y_pred_svm)
print("R2 Score_svm:", r2_score_svm)




MAE_nb : 0.10537790697674419
RMSE_nb: 0.3246196343056658
R2 Score_nb: -0.10437665709082666
MAE_svm : 0.09811046511627906
RMSE_svm: 0.3132259010942088
R2 Score_svm: -0.02821274970525245


## 1.5 过拟合检验

In [29]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

# MultinomialNB
y_train_pred_nb = nb_model.predict(X_train)
print("NB - Training Accuracy:", accuracy_score(y_train, y_train_pred_nb))
print("NB - Test Accuracy:", accuracy_score(y_test, y_pred_nb))
print("NB - Test F1 Score:", f1_score(y_test, y_pred_nb, average='weighted'))

# SVM
y_train_pred_svm = svm_model.predict(X_train)
print("SVM - Training Accuracy:", accuracy_score(y_train, y_train_pred_svm))
print("SVM - Test Accuracy:", accuracy_score(y_test, y_pred_svm))
print("SVM - Test F1 Score:", f1_score(y_test, y_pred_svm, average='weighted'))


NB - Training Accuracy: 0.896965291659095
NB - Test Accuracy: 0.8946220930232558
NB - Test F1 Score: 0.8476213855657289
SVM - Training Accuracy: 0.9165909503906959
SVM - Test Accuracy: 0.9018895348837209
SVM - Test F1 Score: 0.8750697063512449


In [30]:
from sklearn.model_selection import cross_val_score

# MultinomialNB
cv_scores_nb = cross_val_score(nb_model, X_train, y_train, cv=5, scoring='accuracy')
print("NB CV Accuracy:", cv_scores_nb.mean())

# SVM
cv_scores_svm = cross_val_score(svm_model, X_train, y_train, cv=5, scoring='accuracy')
print("SVM CV Accuracy:", cv_scores_svm.mean())


NB CV Accuracy: 0.8896963091404508
SVM CV Accuracy: 0.8951483775080507


# 2. 预测弹幕文本得到初始情绪评分

In [31]:
import os
import pandas as pd
from tqdm import tqdm

results = []
root_folder = "D:/Desktop/Nanjing University/DSAI/每期弹幕数据/每期弹幕"

for week_folder in os.listdir(root_folder):
    week_path = os.path.join(root_folder, week_folder)
    if not os.path.isdir(week_path):
        continue

    print(f"Processing week folder: {week_folder}")
    
    for txt_file in tqdm(os.listdir(week_path)):
        if not txt_file.endswith(".txt"):
            continue
        
        file_path = os.path.join(week_path, txt_file)
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()
        except Exception as e:
            print(f"Failed to read {txt_file}: {e}")
            continue
        
        X_tfidf = vectorizer.transform([text])
        X_selected = selector.transform(X_tfidf)

        # 输出 0~1 的概率
        nb_score = nb_model.predict_proba(X_selected)[0][1]
        svm_score = svm_model.predict_proba(X_selected)[0][1]

        results.append({
            "txt_file": txt_file,
            "week_folder": week_folder,
            "nb_score": nb_score,
            "svm_score": svm_score
        })

df_results = pd.DataFrame(results)
output_path = "D:/Desktop/Nanjing University/DSAI/每期弹幕数据/video_sentiment_scores.xlsx"
df_results.to_excel(output_path, index=False)
print(f"结果已保存到 {output_path}")


Processing week folder: 第100期


100%|██████████| 30/30 [00:00<00:00, 42.98it/s]


Processing week folder: 第101期


100%|██████████| 34/34 [00:00<00:00, 42.54it/s]


Processing week folder: 第102期


...

Processing week folder: 第203期


100%|██████████| 39/39 [00:01<00:00, 33.64it/s]


Processing week folder: 第204期


 62%|██████▏   | 28/45 [00:00<00:00, 34.18it/s]


KeyboardInterrupt: 

In [ ]:
import pandas as pd

# 1. 读取 xlsx 文件
df = pd.read_excel(
    "D:/Desktop/Nanjing University/DSAI/每期弹幕数据/video_sentiment_scores.xlsx"
)

# 2. 从 txt_file 中提取数字作为 cid
df["cid"] = df["txt_file"].str.extract(r"(\d+)\.txt")

# 3. 转为整数（允许缺失）
df["cid"] = df["cid"].astype("Int64")

# 4. 保存为新的 xlsx 文件
save_path = ("D:/Desktop/Nanjing University/DSAI/每期弹幕数据/video_sentiment_scores-1.xlsx")

df.to_excel(save_path, index=False)

# 5. 查看结果
print(df.head())
print(f"文件已成功保存至：{save_path}")




                   txt_file week_folder  nb_score  svm_score        cid
0  第100期第10个视频372193445.txt       第100期  0.000498   0.045288  372193445
1  第100期第11个视频466361744.txt       第100期  0.013056   0.127791  466361744
2  第100期第12个视频298877495.txt       第100期  0.005214   0.120899  298877495
3  第100期第13个视频294705287.txt       第100期  0.000473   0.040938  294705287
4  第100期第14个视频298302821.txt       第100期  0.002894   0.097397  298302821
文件已成功保存至：D:/Desktop/Nanjing University/DSAI/每期弹幕数据/video_sentiment_scores-1.xlsx


# 3. 修正方法一：采用title+describe进行修正

In [ ]:
import pandas as pd

# 1. 读取CSV文件
file_path = r"D:\Desktop\Nanjing University\DSAI\每周必看视频信息.csv"
df = pd.read_csv(file_path)
# 2. 合并title和desc列的文本（处理空值）
df["combined_text"] = df["title"].fillna("") + " " + df["desc"].fillna("")
def get_sentiment_score(text):
    X_tfidf = vectorizer.transform([text])
    X_selected = selector.transform(X_tfidf)

    nb_score = nb_model.predict_proba(X_selected)[0][1]
    svm_score = svm_model.predict_proba(X_selected)[0][1]

    # 两个模型取平均，保证 0–1 小数
    v_score = (nb_score + svm_score) / 2
    return v_score

df["v_score"] = df["combined_text"].apply(get_sentiment_score)

# 4. 筛选需要的列
result_cols = [
    "number", "cid", "pubdate", "view", "danmaku", "reply", 
    "favorite", "coin", "share", "his_rank", "like", "pid_name_v2", "v_score"
]
result_df = df[result_cols]

# 5. 保存为Excel文件（需确保已安装openpyxl）
saved_path = r"D:\Desktop\Nanjing University\DSAI\v_score.xlsx"
result_df.to_excel(saved_path, index=False, engine="openpyxl")

print(f"处理完成！结果已保存至：{saved_path}")

处理完成！结果已保存至：D:\Desktop\Nanjing University\DSAI\v_score.xlsx


In [ ]:
import pandas as pd

# 1. 定义文件路径
video_score_path = r"D:/Desktop/Nanjing University/DSAI/v_score.xlsx"  # 视频情绪评分文件
danmaku_score_path = r"D:/Desktop/Nanjing University/DSAI/每期弹幕数据/video_sentiment_scores-1.xlsx"  # 弹幕情绪评分文件
save_path_1 = r"D:/Desktop/Nanjing University/DSAI/情绪评分-视频+弹幕.xlsx"  # 合并后保存路径

# 2. 读取两个Excel文件（确保列名正确）
df_video = pd.read_excel(video_score_path, engine="openpyxl")
df_danmaku = pd.read_excel(danmaku_score_path, engine="openpyxl")

# 3. 按cid列横向合并（how='inner'仅保留cid都存在的行；若要保留所有行，改为how='outer'）
df_merged = pd.merge(
    df_video,          # 左表：视频评分数据
    df_danmaku,        # 右表：弹幕评分数据
    on="cid",          # 匹配键：cid列
    how="inner",       # 匹配方式：仅保留两边cid都存在的行（按需修改）
    suffixes=("_video", "_danmaku")  # 重复列名的后缀（区分来源）
)

# 4. 处理重复的number列（保留视频表的number，删除弹幕表的number_danmaku）
if "number_danmaku" in df_merged.columns:
    df_merged = df_merged.drop(columns=["number_danmaku"])
# 若视频表的number后缀是number_video，重命名为number（适配merge后列名逻辑）
if "number_video" in df_merged.columns:
    df_merged.rename(columns={"number_video": "number"}, inplace=True)

# 5. 保存合并后的文件
df_merged.to_excel(save_path_1, index=False, engine="openpyxl")

print(f"文件合并完成！结果已保存至：{save_path}")
print(f"合并后数据总行数：{len(df_merged)}")
print(f"合并后数据列名：\n{df_merged.columns.tolist()}")

文件合并完成！结果已保存至：D:/Desktop/Nanjing University/DSAI/每期弹幕数据/video_sentiment_scores-1.xlsx
合并后数据总行数：12851
合并后数据列名：
['number', 'cid', 'pubdate', 'view', 'danmaku', 'reply', 'favorite', 'coin', 'share', 'his_rank', 'like', 'pid_name_v2', 'v_score', 'txt_file', 'week_folder', 'nb_score', 'svm_score']


In [32]:
import pandas as pd
import numpy as np

# ---------------------- 0. 修正函数（保持不变） ----------------------
def correct_emotion_improved(s, v, alpha=0.8, col_name=None):

    global out_of_bounds_stats
    if s == 0.5:
        return 0.5

    dv = v - 0.5
    s_sign = 1 if s > 0.5 else -1
    correction_factor = 1 - alpha * dv * s_sign
    corrected_raw = 0.5 + (s - 0.5) * correction_factor

    if col_name is not None:
        out_of_bounds_stats[col_name]['total'] += 1
        if corrected_raw < 0:
            out_of_bounds_stats[col_name]['below_0'] += 1
        elif corrected_raw > 1:
            out_of_bounds_stats[col_name]['above_1'] += 1

    return np.clip(corrected_raw, 0.0, 1.0)


# ---------------------- 1. 初始化超界统计（两个模型） ----------------------
out_of_bounds_stats = {
    'svm_score': {'below_0': 0, 'above_1': 0, 'total': 0},
    'nb_score':  {'below_0': 0, 'above_1': 0, 'total': 0}
}


# ---------------------- 2. 读取数据并预处理 ----------------------
file_path = r"D:\Desktop\Nanjing University\DSAI\情绪评分-视频+弹幕.xlsx"
df = pd.read_excel(file_path)

print("=== 缺失值统计 ===")
missing_stats = df[['v_score', 'svm_score', 'nb_score']].isnull().sum()
print(missing_stats)

df = df.fillna({
    'v_score': df['v_score'].median(),
    'svm_score': df['svm_score'].median(),
    'nb_score': df['nb_score'].median()
})


# ---------------------- 3. 应用同一个修正函数（两种模型） ----------------------
alpha = 0.8

df['corrected_svm'] = df.apply(
    lambda row: correct_emotion_improved(
        row['svm_score'], row['v_score'], alpha, col_name='svm_score'
    ),
    axis=1
)

df['corrected_nb'] = df.apply(
    lambda row: correct_emotion_improved(
        row['nb_score'], row['v_score'], alpha, col_name='nb_score'
    ),
    axis=1
)


# ---------------------- 4. 超界情况统计 ----------------------
print("\n=== 修正值超界（<0 或 >1）统计结果 ===")
for col in ['svm_score', 'nb_score']:
    stats = out_of_bounds_stats[col]
    total_out = stats['below_0'] + stats['above_1']
    ratio = (total_out / stats['total']) * 100 if stats['total'] > 0 else 0

    print(f"\n【{col}】")
    print(f"  总修正条数：{stats['total']}")
    print(f"  低于0的条数：{stats['below_0']}")
    print(f"  高于1的条数：{stats['above_1']}")
    print(f"  超界总条数：{total_out}（占比：{ratio:.4f}%）")


# ---------------------- 5. 保存结果 ----------------------
output_path = r"D:\Desktop\Nanjing University\DSAI\情绪评分_修正后.xlsx"
df.to_excel(output_path, index=False)

print("\n=== 结果保存完成 ===")
print(f"修正后数据已保存至：{output_path}")
print(df[['v_score', 'svm_score', 'nb_score', 'corrected_svm', 'corrected_nb']].head())


=== 缺失值统计 ===
v_score      0
svm_score    0
nb_score     0
dtype: int64

=== 修正值超界（<0 或 >1）统计结果 ===

【svm_score】
  总修正条数：12819
  低于0的条数：0
  高于1的条数：195
  超界总条数：195（占比：1.5212%）

【nb_score】
  总修正条数：12851
  低于0的条数：3
  高于1的条数：0
  超界总条数：3（占比：0.0233%）

=== 结果保存完成 ===
修正后数据已保存至：D:\Desktop\Nanjing University\DSAI\情绪评分_修正后.xlsx
    v_score  svm_score  nb_score  corrected_svm  corrected_nb
0  0.023151   0.211386  0.007714       0.321486      0.195511
1  0.155585   0.842644  0.132392       0.937054      0.233679
2  0.054002   0.043805  0.001538       0.206575      0.179388
3  0.014672   0.440387  0.005283       0.463532      0.197363
4  0.029463   0.066997  0.000526       0.229992      0.188543


In [33]:
import pandas as pd

# ---------------------- 1. 配置文件路径 ----------------------
# 原始数据路径（r字符串避免转义）
input_file = r"D:\Desktop\Nanjing University\DSAI\情绪评分_修正后.xlsx"
# 月度数据输出路径
output_file = r"D:\Desktop\Nanjing University\DSAI\月度情绪评分数据.xlsx"

# ---------------------- 2. 读取并预处理数据 ----------------------
# 读取Excel文件
df = pd.read_excel(input_file)

# 检查关键列是否存在
required_cols = ['pubdate', 'cid', 'danmaku', 'like', 'v_score', 'svm_score', 'nb_score', 'corrected_svm', 'corrected_nb']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"数据缺失关键列：{missing_cols}")

# 将秒级时间戳转换为datetime格式，提取月份（格式：xxxx-xx）
df['pubdate_dt'] = pd.to_datetime(df['pubdate'], unit='s', errors='coerce')
df['month'] = df['pubdate_dt'].dt.to_period('M').astype(str)

# 过滤无效数据（时间戳转换失败/情绪评分/权重缺失）
df = df.dropna(subset=['month'] + required_cols)
print(f"有效数据行数：{len(df)}，涉及月份数：{df['month'].nunique()}")

# ---------------------- 3. 定义核心列 ----------------------
# 情绪评分列
score_cols = ['v_score', 'svm_score', 'nb_score', 'corrected_svm', 'corrected_nb']
# 权重列（弹幕数、点赞数）
weight_cols = ['danmaku', 'like']

# ---------------------- 4. 计算基础月度统计（数量+均值） ----------------------
# n：每月视频数；score列计算均值
base_stats = df.groupby('month').agg(
    n=('cid', 'count'),  # 每月视频数量
    **{col: (col, 'mean') for col in score_cols}  # 各评分月度均值
).reset_index()

# ---------------------- 5. 计算加权月度评分（弹幕/点赞为权重） ----------------------
# 初始化加权统计结果容器
weighted_results = []

# 遍历每个月份，计算所有加权评分
for month in df['month'].unique():
    month_data = df[df['month'] == month].copy()
    res = {'month': month}
    
    # 遍历权重列和评分列，计算加权均值
    for weight_col in weight_cols:
        for score_col in score_cols:
            # 过滤有效数据，避免空值影响
            valid = month_data[[score_col, weight_col]].dropna()
            total_weight = valid[weight_col].sum()
            
            # 计算加权均值（权重和为0时返回0）
            if total_weight == 0:
                weighted_val = 0.0
            else:
                weighted_val = (valid[score_col] * valid[weight_col]).sum() / total_weight
            
            res[f"{score_col}_weighted_by_{weight_col}"] = weighted_val
    
    weighted_results.append(res)

# 转换为DataFrame
weighted_stats = pd.DataFrame(weighted_results)

# ---------------------- 6. 合并数据并整理 ----------------------
# 合并基础统计和加权统计
final_df = pd.merge(base_stats, weighted_stats, on='month', how='left')

# 按月份排序
final_df['month_dt'] = pd.to_datetime(final_df['month'])
final_df = final_df.sort_values('month_dt').drop('month_dt', axis=1).reset_index(drop=True)

# ---------------------- 7. 保存 ----------------------
final_df.to_excel(output_file, index=False)


有效数据行数：12851，涉及月份数：81


# 3. 修正方法二：采用同标签视频进行中心化

In [34]:
import pandas as pd

# 读取数据
df = pd.read_excel(r"D:\Desktop\Nanjing University\DSAI\情绪评分-视频+弹幕.xlsx")

# ----------------------
# 1. 计算组内均值（不改变行数）
# ----------------------
df['svm_mean_pid'] = df.groupby('pid_name_v2')['svm_score'].transform('mean')
df['nb_mean_pid']  = df.groupby('pid_name_v2')['nb_score'].transform('mean')

# ----------------------
# 2. 组内中心化（demeaning）
# ----------------------
df['svm_centered'] = df['svm_score'] - df['svm_mean_pid']
df['nb_centered']  = df['nb_score']  - df['nb_mean_pid']

# ----------------------
# 3. 保存结果
# ----------------------
output_path = r"D:\Desktop\Nanjing University\DSAI\情绪评分_修正后二.xlsx"
df.to_excel(output_path, index=False)

print("处理完成，前5行结果：")
print(df[['pid_name_v2',
          'svm_score', 'svm_mean_pid', 'svm_centered',
          'nb_score',  'nb_mean_pid',  'nb_centered']].head())


处理完成，前5行结果：
  pid_name_v2  svm_score  svm_mean_pid  svm_centered  nb_score  nb_mean_pid  \
0        vlog   0.211386      0.226380     -0.014994  0.007714     0.038624   
1          影视   0.842644      0.128172      0.714473  0.132392     0.017021   
2         二次元   0.043805      0.133382     -0.089576  0.001538     0.019561   
3          美食   0.440387      0.141056      0.299331  0.005283     0.019170   
4          游戏   0.066997      0.116895     -0.049898  0.000526     0.021282   

   nb_centered  
0    -0.030910  
1     0.115371  
2    -0.018024  
3    -0.013886  
4    -0.020757  


In [2]:
import pandas as pd

# ---------------------- 1. 配置文件路径 ----------------------
# 原始数据路径（r字符串避免转义）
input_file = r"D:\Desktop\Nanjing University\DSAI\dataset\情绪评分_再修正后二.xlsx"
# 月度数据输出路径
output_file = r"D:\Desktop\Nanjing University\DSAI\dataset\月度情绪评分数据二（再修正）.xlsx"

# ---------------------- 2. 读取并预处理数据 ----------------------
# 读取Excel文件
df = pd.read_excel(input_file)

# 检查关键列是否存在
required_cols = ['pubdate', 'cid', 'danmaku', 'like', 'v_score', 'svm_score', 'nb_score', 'svm_centered', 'nb_centered', 'svm_corrected2_score', 'nb_corrected2_score']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"数据缺失关键列：{missing_cols}")

# 将秒级时间戳转换为datetime格式，提取月份（格式：xxxx-xx）
df['pubdate_dt'] = pd.to_datetime(df['pubdate'], unit='s', errors='coerce')
df['month'] = df['pubdate_dt'].dt.to_period('M').astype(str)

# 过滤无效数据（时间戳转换失败/情绪评分/权重缺失）
df = df.dropna(subset=['month'] + required_cols)
print(f"有效数据行数：{len(df)}，涉及月份数：{df['month'].nunique()}")

# ---------------------- 3. 定义核心列 ----------------------
# 情绪评分列
score_cols = ['v_score', 'svm_score', 'nb_score', 'svm_centered', 'nb_centered', 'svm_corrected2_score', 'nb_corrected2_score']
# 权重列（弹幕数、点赞数）
weight_cols = ['danmaku']

# ---------------------- 4. 计算基础月度统计（数量+均值） ----------------------
# n：每月视频数；score列计算均值
base_stats = df.groupby('month').agg(
    n=('cid', 'count'),  # 每月视频数量
    **{col: (col, 'mean') for col in score_cols}  # 各评分月度均值
).reset_index()

# ---------------------- 5. 计算加权月度评分（弹幕/点赞为权重） ----------------------
# 初始化加权统计结果容器
weighted_results = []

# 遍历每个月份，计算所有加权评分
for month in df['month'].unique():
    month_data = df[df['month'] == month].copy()
    res = {'month': month}
    
    # 遍历权重列和评分列，计算加权均值
    for weight_col in weight_cols:
        for score_col in score_cols:
            # 过滤有效数据，避免空值影响
            valid = month_data[[score_col, weight_col]].dropna()
            total_weight = valid[weight_col].sum()
            
            # 计算加权均值（权重和为0时返回0）
            if total_weight == 0:
                weighted_val = 0.0
            else:
                weighted_val = (valid[score_col] * valid[weight_col]).sum() / total_weight
            
            res[f"{score_col}_weighted_by_{weight_col}"] = weighted_val
    
    weighted_results.append(res)

# 转换为DataFrame
weighted_stats = pd.DataFrame(weighted_results)

# ---------------------- 6. 合并数据并整理 ----------------------
# 合并基础统计和加权统计
final_df = pd.merge(base_stats, weighted_stats, on='month', how='left')

# 按月份排序
final_df['month_dt'] = pd.to_datetime(final_df['month'])
final_df = final_df.sort_values('month_dt').drop('month_dt', axis=1).reset_index(drop=True)

# ---------------------- 7. 保存 ----------------------
final_df.to_excel(output_file, index=False)

有效数据行数：12851，涉及月份数：81
